# Solution 6 — RAG generation WITH the reranker

**Why this run exists.** Every generation result so far used plain e5 retrieval. Reranking was discovered afterward and is the strongest result in the project (Darija R@1: 0.620 → 0.800, gap nearly halved). This checks whether that gain **reaches the final answer**, not just the ranking.

| | Condition |
|---|---|
| **C1** | MSA query, e5 retrieval (ceiling) |
| **C2** | Darija query, e5 retrieval (mismatch — same seed as the earlier n=200 run, for comparison) |
| **C5** | Darija query, e5 retrieval **+ bge-reranker-v2-m3** (the new condition) |
| **C4** | gold passage given directly (oracle) |

**Also builds a combined judge-validation sheet** — 50 items from this run + 50 from your earlier n=200 run in one sheet, so one labeling session validates the correctness judge across both retrieval setups.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`, and optionally `generation_n200_raw.csv` (from the earlier run) for the combined validation. **GPU required.**

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers transformers accelerate bitsandbytes

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "prior_run_csv": "generation_n200_raw.csv",  # optional; set to None if you don't have it

    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,
    "retrieve_k": 20,      # candidates passed to the reranker
    "final_k": 5,           # passages actually given to the generator, after reranking
    "reranker": "BAAI/bge-reranker-v2-m3",

    "n_eval": 200,
    "llm": "Qwen/Qwen2.5-7B-Instruct",
    "load_4bit": True,
    "max_new_tokens": 128,
    "batch_size": 8,

    "checkpoint": "gen_reranked_checkpoint.json",
    "seed": 42,
}

### Load data

In [ ]:
import json, random, re, os, gc
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
random.Random(CONFIG["seed"]).shuffle(qa)
eval_qa = qa[: CONFIG["n_eval"]]
print(f"Corpus {len(corpus)} | evaluating {len(eval_qa)} (same seed as the earlier n=200 run)")

### BM25 + Arabic normalization

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Stage 1: retrieve top-K candidates for every condition

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("Building retrieval index...")
bi = SentenceTransformer(CONFIG["base_encoder"])
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")

def retrieve_candidates(query, k):
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = CONFIG["alpha"] * minmax(corpus_emb @ q) + (1 - CONFIG["alpha"]) * minmax(bm25_scores(query))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

K = CONFIG["retrieve_k"]
raw_candidates = {}
for field in ["msa_query", "darija_query"]:
    raw_candidates[field] = {q["id"]: retrieve_candidates(q[field], K) for q in eval_qa}
    print(f"  {field}: top-{K} retrieved")

del bi, corpus_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Stage 2: rerank the Darija candidates, then free the reranker

In [ ]:
from sentence_transformers import CrossEncoder

print(f"Loading reranker: {CONFIG['reranker']}")
ce = CrossEncoder(CONFIG["reranker"], max_length=512, trust_remote_code=True,
                  automodel_args={"torch_dtype": torch.float32})

reranked_darija = {}
for q in eval_qa:
    cands = raw_candidates["darija_query"][q["id"]]
    pairs = [(q["darija_query"], corpus_map[c]) for c in cands]
    scores = ce.predict(pairs, batch_size=16, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))
    reranked_darija[q["id"]] = [cands[i] for i in order]

del ce
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Reranking complete.")

### Build final top-k contexts for all four conditions

In [ ]:
FK = CONFIG["final_k"]
contexts = {
    "C1_msa_e5":       {q["id"]: raw_candidates["msa_query"][q["id"]][:FK] for q in eval_qa},
    "C2_darija_e5":    {q["id"]: raw_candidates["darija_query"][q["id"]][:FK] for q in eval_qa},
    "C5_darija_reranked": {q["id"]: reranked_darija[q["id"]][:FK] for q in eval_qa},
    "C4_oracle":       {q["id"]: [q["source_chunk_id"]] for q in eval_qa},
}

for cond, d in contexts.items():
    hit = np.mean([q["source_chunk_id"] in d[q["id"]] for q in eval_qa])
    print(f"  {cond:<22} gold in top-{FK}: {hit:.3f}")

### Load the local LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
) if CONFIG["load_4bit"] else None

tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

llm = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm"], quantization_config=quant, device_map="auto", torch_dtype=torch.float16)
llm.eval()
print(f"Loaded {CONFIG['llm']}")

@torch.no_grad()
def chat_batch(prompts, max_new_tokens=None):
    mnt = max_new_tokens or CONFIG["max_new_tokens"]
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=3072).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=mnt, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tok.decode(g, skip_special_tokens=True).strip() for g in gen]

print("Smoke test:", chat_batch(["أجب بكلمة واحدة: ما عاصمة المغرب؟"], 20)[0])

### Prompts (Arabic-only constraint, the one that measurably helped earlier)

In [ ]:
GEN_PROMPT = """أجب عن السؤال التالي اعتمادا فقط على النصوص المرفقة.

قواعد إلزامية:
- أجب بالعربية فقط. ممنوع استعمال أي كلمة بحرف لاتيني.
- إذا لم تكن الإجابة موجودة في النصوص، اكتب بالضبط: المعلومة غير متوفرة في النصوص
- لا تستعمل أي معرفة خارجية.
- أجب بجملة واحدة قصيرة فقط.

النصوص:
{context}

السؤال: {question}

الإجابة:"""

JUDGE_PROMPT = """النصوص المرجعية:
{context}

السؤال: {question}
الإجابة الصحيحة: {gold}
الإجابة المقدمة: {answer}

أجب عن سؤالين بدقة:
1. هل كل ما ورد في الإجابة المقدمة مدعوم صراحة بالنصوص المرجعية؟
2. هل الإجابة المقدمة مطابقة في المعنى للإجابة الصحيحة؟ اختلاف الصياغة مقبول، أما اختلاف الأرقام أو الأسماء أو التواريخ فغير مقبول.

أجب بهذا الشكل فقط وبدون أي شرح:
مدعوم: نعم/لا
مطابق: نعم/لا"""

def ctx_text(chunk_ids):
    return "\n\n".join(f"[{i+1}] {corpus_map[c]}" for i, c in enumerate(chunk_ids))

def parse_judge(text):
    t = (text or "").replace("،", " ")
    faithful = correct = 0
    for line in t.split("\n"):
        if "مدعوم" in line:
            faithful = 1 if "نعم" in line else 0
        elif "مطابق" in line:
            correct = 1 if "نعم" in line else 0
    return faithful, correct

CONDITION_QUERY = {
    "C1_msa_e5": "msa_query", "C2_darija_e5": "darija_query",
    "C5_darija_reranked": "darija_query", "C4_oracle": "darija_query",
}

### Run generation + judging (batched + checkpointed)

In [ ]:
from tqdm.auto import tqdm

records = []
if os.path.exists(CONFIG["checkpoint"]):
    records = json.load(open(CONFIG["checkpoint"], encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["qid"], r["condition"]) for r in records}
byid = {q["id"]: q for q in eval_qa}
B = CONFIG["batch_size"]

for cond, ctx_map in contexts.items():
    qfield = CONDITION_QUERY[cond]
    todo = [q for q in eval_qa if (q["id"], cond) not in done]
    if not todo:
        continue
    print(f"\n=== {cond} ({len(todo)} to do) ===")
    for i in tqdm(range(0, len(todo), B)):
        batch = todo[i:i + B]
        chunks = [ctx_map[q["id"]] for q in batch]
        answers = chat_batch([GEN_PROMPT.format(context=ctx_text(c), question=q[qfield])
                              for q, c in zip(batch, chunks)])
        verdicts = chat_batch([JUDGE_PROMPT.format(context=ctx_text(c), question=q["msa_query"],
                                                   gold=q["gold_answer"], answer=a)
                               for q, c, a in zip(batch, chunks, answers)], 40)
        for q, c, a, v in zip(batch, chunks, answers, verdicts):
            f, ok = parse_judge(v)
            records.append({
                "qid": q["id"], "condition": cond,
                "gold_in_context": int(q["source_chunk_id"] in c),
                "answer": a, "faithful": f, "correct": ok,
                "refused": int("غير متوفرة" in (a or "")),
            })
        json.dump(records, open(CONFIG["checkpoint"], "w", encoding="utf-8"), ensure_ascii=False, indent=2)

gen = pd.DataFrame(records)
gen.to_csv("generation_reranked_raw.csv", index=False)
print(f"\nComplete: {len(gen)} generations.")

### Results: does the reranker's retrieval gain reach the answers?

In [ ]:
print("=" * 90)
print("RESULTS BY CONDITION")
print("=" * 90)
order = [c for c in CONDITION_QUERY if c in gen.condition.unique()]
summary = gen.groupby("condition").agg(
    n=("qid", "count"), gold_in_context=("gold_in_context", "mean"),
    faithfulness=("faithful", "mean"), correctness=("correct", "mean"),
    refusal_rate=("refused", "mean"),
).reindex(order)
print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary.to_csv("generation_reranked_summary.csv")

### The connecting comparison, with paired bootstrap CIs

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a_cond, b_cond, col):
    a = gen[gen.condition == a_cond].set_index("qid")[col]
    b = gen[gen.condition == b_cond].set_index("qid")[col]
    common = a.index.intersection(b.index)
    d = (a.loc[common] - b.loc[common]).values.astype(float)
    idx = rng.integers(0, len(d), size=(1000, len(d)))
    m = d[idx].mean(axis=1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi

print("\n" + "=" * 90)
print("DOES THE RERANKER'S RETRIEVAL GAIN REACH THE FINAL ANSWERS?")
print("=" * 90)
comps = [
    ("C1_msa_e5", "C2_darija_e5", "Dialect gap WITHOUT reranking (replicates the earlier n=200 result)"),
    ("C5_darija_reranked", "C2_darija_e5", "Does reranking improve the ANSWER, not just the ranking?"),
    ("C1_msa_e5", "C5_darija_reranked", "Residual gap AFTER reranking"),
]
rows = []
for a, b, label in comps:
    if a not in gen.condition.unique() or b not in gen.condition.unique():
        continue
    print(f"\n{label}   [{a} - {b}]")
    for col in ["correct", "faithful"]:
        d, lo, hi = paired(a, b, col)
        sig = "yes" if (lo > 0 or hi < 0) else "no"
        print(f"  {col:<10} {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  significant: {sig}")
        rows.append({"comparison": f"{a} vs {b}", "metric": col, "diff": d, "lo": lo, "hi": hi, "significant": sig})
pd.DataFrame(rows).to_csv("generation_reranked_comparisons.csv", index=False)

### Build the COMBINED correctness-validation sheet (both rounds at once)

In [ ]:
print("\n" + "=" * 90)
print("BUILDING COMBINED JUDGE-VALIDATION SHEET (this run + the earlier n=200 run)")
print("=" * 90)

def stratified_sample(df, n, qa_lookup, run_label):
    n_per = max(1, n // df["condition"].nunique())
    parts = []
    for cond, grp in df.groupby("condition"):
        wrong = grp[grp["correct"] == 0]
        right = grp[grp["correct"] == 1]
        n_wrong = min(len(wrong), max(1, n_per // 2))
        n_right = min(len(right), n_per - n_wrong)
        parts.append(pd.concat([
            wrong.sample(n_wrong, random_state=42) if n_wrong else wrong,
            right.sample(n_right, random_state=42) if n_right else right,
        ]))
    sample = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True).head(n)
    rows = []
    for _, r in sample.iterrows():
        q = qa_lookup.get(r["qid"], {})
        rows.append({
            "run": run_label, "qid": r["qid"], "condition": r["condition"],
            "msa_query": q.get("msa_query", ""), "darija_query": q.get("darija_query", ""),
            "gold_answer": q.get("gold_answer", ""), "model_answer": r.get("answer", ""),
            "llm_correct": int(r["correct"]), "human_correct": "",
        })
    return pd.DataFrame(rows)

qa_lookup = {q["id"]: q for q in eval_qa}
sheets = [stratified_sample(gen, 50, qa_lookup, "reranked_run")]

if CONFIG["prior_run_csv"] and os.path.exists(CONFIG["prior_run_csv"]):
    prior = pd.read_csv(CONFIG["prior_run_csv"])
    sheets.append(stratified_sample(prior, 50, qa_lookup, "n200_run"))
    print(f"Included {CONFIG['prior_run_csv']} -- combined sheet covers BOTH rounds.")
else:
    print(f"'{CONFIG['prior_run_csv']}' not found -- sheet covers only this run. "
          f"Upload the earlier CSV and rerun this cell to validate both rounds together.")

combined = pd.concat(sheets, ignore_index=True)
combined.to_csv("combined_labeling_sheet.csv", index=False, encoding="utf-8-sig")
print(f"\nWrote {len(combined)} rows to combined_labeling_sheet.csv")
print(f"  reranked_run: {(combined['run']=='reranked_run').sum()}")
print(f"  n200_run:     {(combined['run']=='n200_run').sum()}")
print("\nNext: label 'human_correct' with 0/1 for every row, then run")
print("    python validate_judge.py score --sheet combined_labeling_sheet.csv")
print("(validate_judge.py's 'score' command works on this file as-is -- same columns.)")

from google.colab import files
files.download("generation_reranked_raw.csv")
files.download("generation_reranked_summary.csv")
files.download("generation_reranked_comparisons.csv")
files.download("combined_labeling_sheet.csv")